# Step 8: Create Combined EEG Dataset

Combine the three harmonized, preprocessed, quality-controlled, and segmented datasets into one standardized dataset.

Rules:
- Use only quality-approved segments
- No raw EEG files
- No model training, no augmentation, no normalization
- No random segment-level train/test split (must be subject-level later)
- Preserve subject identity
- Keep MCI as separate label

In [1]:
%pip install --force-reinstall "numpy==1.24.4" "pandas==1.5.3" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.4 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 which is incompatible.
conda-repo-cli 1.0.20 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.20 requires requests==2.28.1, but you have requests 2.32.5 which is incompatible.
botocore 1.27.28 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 200)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'NOTEBOOK':
    NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NOTEBOOK' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
SEG_DIR = PROCESSED_DIR / 'segmentation'
COMBINED_DIR = PROCESSED_DIR / 'combined'
SEG_META_PATH = SEG_DIR / 'segment_metadata.csv'
COMBINED_META_PATH = COMBINED_DIR / 'combined_metadata.csv'
DATASET_SUMMARY_PATH = COMBINED_DIR / 'dataset_summary.csv'
CLASS_SUMMARY_PATH = COMBINED_DIR / 'class_summary.csv'
SUBJECT_SUMMARY_PATH = COMBINED_DIR / 'subject_summary.csv'
SEG_FILES_DIR = COMBINED_DIR / 'eeg_segments'

COMBINED_DIR.mkdir(parents=True, exist_ok=True)
SEG_FILES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CHANNELS = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
    'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3',
    'Pz', 'P4', 'T6', 'O1', 'O2'
]
TARGET_SFREQ = 250.0
SEG_SAMPLES = 7500
SEG_SHAPE = (19, 7500)

print('Combined dataset creation started')

Combined dataset creation started


## 1. Load Segment Metadata

In [3]:
seg_meta = pd.read_csv(SEG_META_PATH)
print(f'Loaded {len(seg_meta)} segment records')
print(f'Datasets: {seg_meta["dataset"].unique()}')
print(f'Labels: {seg_meta["standard_label"].unique()}')
print(f'Subjects: {seg_meta["subject_id"].nunique()}')
print(f'Columns: {list(seg_meta.columns)}')

Loaded 6381 segment records
Datasets: ['ADFTD' 'AD_Auditory']
Labels: ['AD' 'HC' 'FTD' 'UNKNOWN' 'MCI']
Subjects: 88
Columns: ['segment_id', 'dataset', 'subject_id', 'standard_label', 'source_file', 'start_time', 'end_time', 'sampling_rate', 'channels', 'num_samples']


## 2. Verify Segment Files and Data Integrity

In [4]:
def verify_segment_file(row):
    """Verify a single segment file."""
    seg_file = Path(row.get('segment_file', ''))
    if not seg_file.is_absolute():
        seg_file = SEG_DIR / row['dataset'] / f'{row["subject_id"]}_seg{row["segment_id"]:06d}.npy'
    
    result = {
        'global_segment_id': row['segment_id'],
        'file_exists': False,
        'shape_ok': False,
        'no_nan': False,
        'no_inf': False,
        'error': ''
    }
    
    if not seg_file.exists():
        result['error'] = 'file_not_found'
        return result
    
    result['file_exists'] = True
    
    try:
        data = np.load(seg_file)
        
        if data.shape == SEG_SHAPE:
            result['shape_ok'] = True
        else:
            result['error'] = f'shape_mismatch: {data.shape} != {SEG_SHAPE}'
        
        if not np.any(np.isnan(data)):
            result['no_nan'] = True
        else:
            result['error'] = (result['error'] + '; ' if result['error'] else '') + 'contains_nan'
        
        if not np.any(np.isinf(data)):
            result['no_inf'] = True
        else:
            result['error'] = (result['error'] + '; ' if result['error'] else '') + 'contains_inf'
        
    except Exception as exc:
        result['error'] = f'load_error: {exc}'
    
    return result

In [5]:
print('Verifying segment files...')
verify_results = []
for idx, row in seg_meta.iterrows():
    if idx % 1000 == 0:
        print(f'  {idx+1}/{len(seg_meta)}')
    result = verify_segment_file(row)
    verify_results.append(result)

verify_df = pd.DataFrame(verify_results)
print(f'\nVerification results:')
print(f"  File exists: {verify_df['file_exists'].sum()}/{len(verify_df)}")
print(f"  Shape OK: {verify_df['shape_ok'].sum()}/{len(verify_df)}")
print(f"  No NaN: {verify_df['no_nan'].sum()}/{len(verify_df)}")
print(f"  No Inf: {verify_df['no_inf'].sum()}/{len(verify_df)}")

failed = verify_df[~verify_df['file_exists'] | ~verify_df['shape_ok'] | ~verify_df['no_nan'] | ~verify_df['no_inf']]
if len(failed) > 0:
    print(f'\n{len(failed)} segments failed verification:')
    for _, r in failed.head(20).iterrows():
        print(f"  global_segment_id={r['global_segment_id']}: {r['error']}")
else:
    print('\nAll segments passed verification!')

Verifying segment files...
  1/6381
  1001/6381
  2001/6381
  3001/6381
  4001/6381
  5001/6381
  6001/6381

Verification results:
  File exists: 6381/6381
  Shape OK: 6381/6381
  No NaN: 6381/6381
  No Inf: 6381/6381

All segments passed verification!


## 3. Create Master Metadata Table with Global Segment IDs

In [6]:
master_meta = seg_meta.copy().reset_index(drop=True)
master_meta['global_segment_id'] = master_meta.index + 1

if 'segment_file' not in master_meta.columns:
    master_meta['segment_file'] = master_meta.apply(
        lambda r: str(SEG_DIR / r['dataset'] / f'{r["subject_id"]}_seg{r["segment_id"]:06d}.npy'),
        axis=1
    )

master_meta = master_meta.rename(columns={'segment_id': 'local_segment_id'})

master_meta = master_meta[[
    'global_segment_id',
    'dataset',
    'subject_id',
    'standard_label',
    'source_file',
    'segment_file',
    'start_time',
    'end_time',
    'sampling_rate',
    'channels',
    'num_samples'
]]

print(f'Master metadata shape: {master_meta.shape}')
print(f'Global segment IDs: 1 to {master_meta["global_segment_id"].max()}')
print(f'Unique global_segment_id: {master_meta["global_segment_id"].nunique()}')
print(f'Unique subject_id: {master_meta["subject_id"].nunique()}')

Master metadata shape: (6381, 11)
Global segment IDs: 1 to 6381
Unique global_segment_id: 6381
Unique subject_id: 88


## 4. Data Integrity Checks

In [7]:
print('=== DATA INTEGRITY CHECKS ===')

sr_check = master_meta['sampling_rate'].unique()
print(f'2. Sampling rates: {sr_check}')
assert all(sr == TARGET_SFREQ for sr in sr_check), 'Invalid sampling rate'

print('3. NaN/Inf: verified in step 2')
assert master_meta['subject_id'].notna().all()

valid_labels = {'AD', 'FTD', 'HC', 'MCI', 'UNKNOWN'}
invalid_labels = set(master_meta['standard_label'].unique()) - valid_labels
print(f'5. Valid labels: {len(invalid_labels) == 0} (invalid: {invalid_labels})')
assert len(invalid_labels) == 0

valid_datasets = {'ADFTD', 'BrainLat', 'AD_Auditory'}
invalid_datasets = set(master_meta['dataset'].unique()) - valid_datasets
print(f'6. Valid datasets: {len(invalid_datasets) == 0} (invalid: {invalid_datasets})')
assert len(invalid_datasets) == 0

file_exists = master_meta['segment_file'].apply(Path).apply(lambda p: p.exists())
print(f'7. Segment files exist: {file_exists.all()} ({file_exists.sum()}/{len(file_exists)})')
assert file_exists.all()

expected_channels = ';'.join(TARGET_CHANNELS)
channel_check = (master_meta['channels'] == expected_channels).all()
print(f'8. Channel order correct: {channel_check}')
if not channel_check:
    print(f'   Expected: {expected_channels}')
    print(f'   Found: {master_meta["channels"].unique()}')

dup_gsid = master_meta['global_segment_id'].duplicated().sum()
print(f'9. Duplicate global_segment_id: {dup_gsid}')
assert dup_gsid == 0

dup_files = master_meta['segment_file'].duplicated().sum()
print(f'10. Duplicate segment file refs: {dup_files}')
assert dup_files == 0

subj_labels = master_meta.groupby('subject_id')['standard_label'].nunique()
conflicting = subj_labels[subj_labels > 1]
print(f'11. Subjects with conflicting labels: {len(conflicting)}')
if len(conflicting) > 0:
    for subj in conflicting.index:
        labels = master_meta[master_meta['subject_id'] == subj]['standard_label'].unique()
        print(f'   {subj}: {labels}')

print('\nAll integrity checks passed!')

=== DATA INTEGRITY CHECKS ===
2. Sampling rates: [250.]
3. NaN/Inf: verified in step 2
5. Valid labels: True (invalid: set())
6. Valid datasets: True (invalid: set())
7. Segment files exist: True (6381/6381)
8. Channel order correct: True
9. Duplicate global_segment_id: 0
10. Duplicate segment file refs: 0
11. Subjects with conflicting labels: 0

All integrity checks passed!


## 5. Create Summary Reports

In [8]:
ds_summary_rows = []
for ds in ['ADFTD', 'BrainLat', 'AD_Auditory']:
    subset = master_meta[master_meta['dataset'] == ds]
    ds_summary_rows.append({
        'dataset': ds,
        'subjects': subset['subject_id'].nunique(),
        'recordings': subset['source_file'].nunique() if 'source_file' in subset.columns else subset['subject_id'].nunique(),
        'segments': len(subset),
        'AD_segments': len(subset[subset['standard_label'] == 'AD']),
        'FTD_segments': len(subset[subset['standard_label'] == 'FTD']),
        'HC_segments': len(subset[subset['standard_label'] == 'HC']),
        'MCI_segments': len(subset[subset['standard_label'] == 'MCI']),
    })

dataset_summary = pd.DataFrame(ds_summary_rows)
dataset_summary.to_csv(DATASET_SUMMARY_PATH, index=False)
print('DATASET SUMMARY:')
print(dataset_summary.to_string(index=False))

DATASET SUMMARY:
    dataset  subjects  recordings  segments  AD_segments  FTD_segments  HC_segments  MCI_segments
      ADFTD        57         114      5848         2460          1536         1852             0
   BrainLat         0           0         0            0             0            0             0
AD_Auditory        31          31       533          223             0          166           114


In [9]:
class_summary_rows = []
for label in ['AD', 'FTD', 'HC', 'MCI', 'UNKNOWN']:
    subset = master_meta[master_meta['standard_label'] == label]
    if len(subset) == 0:
        class_summary_rows.append({
            'standard_label': label,
            'unique_subjects': 0,
            'total_segments': 0,
            'mean_segments_per_subject': 0,
            'min_segments_per_subject': 0,
            'max_segments_per_subject': 0,
        })
    else:
        segs_per_subj = subset.groupby('subject_id').size()
        class_summary_rows.append({
            'standard_label': label,
            'unique_subjects': subset['subject_id'].nunique(),
            'total_segments': len(subset),
            'mean_segments_per_subject': segs_per_subj.mean(),
            'min_segments_per_subject': segs_per_subj.min(),
            'max_segments_per_subject': segs_per_subj.max(),
        })

class_summary = pd.DataFrame(class_summary_rows)
class_summary.to_csv(CLASS_SUMMARY_PATH, index=False)
print('CLASS SUMMARY:')
print(class_summary.to_string(index=False))

CLASS SUMMARY:
standard_label  unique_subjects  total_segments  mean_segments_per_subject  min_segments_per_subject  max_segments_per_subject
            AD               36            2683                  74.527778                        11                       172
           FTD               17            1536                  90.352941                        60                       124
            HC               27            2018                  74.740741                        11                       128
           MCI                6             114                  19.000000                        19                        19
       UNKNOWN                2              30                  15.000000                        11                        19


In [10]:
subject_summary = master_meta.groupby(['dataset', 'subject_id', 'standard_label']).size().reset_index(name='number_of_segments')
subject_summary = subject_summary.sort_values(['dataset', 'subject_id']).reset_index(drop=True)
subject_summary.to_csv(SUBJECT_SUMMARY_PATH, index=False)
print(f'SUBJECT SUMMARY ({len(subject_summary)} subjects):')
print(subject_summary.to_string(index=False))

SUBJECT SUMMARY (88 subjects):
    dataset subject_id standard_label  number_of_segments
      ADFTD    sub-001             AD                  76
      ADFTD    sub-002             AD                 104
      ADFTD    sub-003             AD                  40
      ADFTD    sub-004             AD                  92
      ADFTD    sub-005             AD                 104
      ADFTD    sub-006             AD                  84
      ADFTD    sub-007             AD                 100
      ADFTD    sub-009             AD                  80
      ADFTD    sub-010             AD                 172
      ADFTD    sub-015             AD                 120
      ADFTD    sub-018             AD                 112
      ADFTD    sub-019             AD                 120
      ADFTD    sub-020             AD                 112
      ADFTD    sub-021             AD                 120
      ADFTD    sub-024             AD                 100
      ADFTD    sub-026             AD    

## 6. Final Dataset Report

In [11]:
print('='*60)
print('FINAL COMBINED DATASET REPORT')
print('='*60)

total_subjects = master_meta['subject_id'].nunique()
total_recordings = master_meta['source_file'].nunique()
total_segments = len(master_meta)

ad_subjects = master_meta[master_meta['standard_label'] == 'AD']['subject_id'].nunique()
ftd_subjects = master_meta[master_meta['standard_label'] == 'FTD']['subject_id'].nunique()
hc_subjects = master_meta[master_meta['standard_label'] == 'HC']['subject_id'].nunique()
mci_subjects = master_meta[master_meta['standard_label'] == 'MCI']['subject_id'].nunique()
unknown_subjects = master_meta[master_meta['standard_label'] == 'UNKNOWN']['subject_id'].nunique()

ad_segments = len(master_meta[master_meta['standard_label'] == 'AD'])
ftd_segments = len(master_meta[master_meta['standard_label'] == 'FTD'])
hc_segments = len(master_meta[master_meta['standard_label'] == 'HC'])
mci_segments = len(master_meta[master_meta['standard_label'] == 'MCI'])
unknown_segments = len(master_meta[master_meta['standard_label'] == 'UNKNOWN'])

print(f'Total unique subjects: {total_subjects}')
print(f'Total EEG recordings: {total_recordings}')
print(f'Total EEG segments: {total_segments}')
print(f'\nTotal AD subjects: {ad_subjects}')
print(f'Total FTD subjects: {ftd_subjects}')
print(f'Total HC subjects: {hc_subjects}')
print(f'Total MCI subjects: {mci_subjects}')
print(f'Total UNKNOWN subjects: {unknown_subjects}')
print(f'\nTotal AD segments: {ad_segments}')
print(f'Total FTD segments: {ftd_segments}')
print(f'Total HC segments: {hc_segments}')
print(f'Total MCI segments: {mci_segments}')
print(f'Total UNKNOWN segments: {unknown_segments}')

print(f'\nCombined EEG shape: ({total_segments}, 19, 7500)')

print('\nDataset composition:')
for ds in ['ADFTD', 'BrainLat', 'AD_Auditory']:
    subset = master_meta[master_meta['dataset'] == ds]
    print(f'  {ds}: {subset["subject_id"].nunique()} subjects, {len(subset)} segments')
    if len(subset) > 0:
        print(f'    Labels: {subset["standard_label"].value_counts().to_dict()}')

print('\nSegment distribution per subject:')
segs_per_subj = master_meta.groupby('subject_id').size()
print(f'  Min: {segs_per_subj.min()}')
print(f'  Max: {segs_per_subj.max()}')
print(f'  Mean: {segs_per_subj.mean():.1f}')
print(f'  Median: {segs_per_subj.median():.1f}')

print('\nIMPORTANT NOTES:')
print('  - Datasets are NOT statistically identical')
print('  - Combining does NOT remove dataset bias')
print('  - AD-Auditory is task-based (40Hz auditory), ADFTD/BrainLat are resting-state')
print('  - BrainLat 128-channel data not yet included (needs 128->19 mapping)')
print('  - MCI kept as separate class (not merged with AD)')
print('  - Subject-wise train/val/test split required to prevent leakage')

print('\nCOMBINED EEG DATASET CREATED — READY FOR SUBJECT-WISE TRAIN/VALIDATION/TEST SPLIT')

FINAL COMBINED DATASET REPORT
Total unique subjects: 88
Total EEG recordings: 145
Total EEG segments: 6381

Total AD subjects: 36
Total FTD subjects: 17
Total HC subjects: 27
Total MCI subjects: 6
Total UNKNOWN subjects: 2

Total AD segments: 2683
Total FTD segments: 1536
Total HC segments: 2018
Total MCI segments: 114
Total UNKNOWN segments: 30

Combined EEG shape: (6381, 19, 7500)

Dataset composition:
  ADFTD: 57 subjects, 5848 segments
    Labels: {'AD': 2460, 'HC': 1852, 'FTD': 1536}
  BrainLat: 0 subjects, 0 segments
  AD_Auditory: 31 subjects, 533 segments
    Labels: {'AD': 223, 'HC': 166, 'MCI': 114, 'UNKNOWN': 30}

Segment distribution per subject:
  Min: 11
  Max: 172
  Mean: 72.5
  Median: 86.0

IMPORTANT NOTES:
  - Datasets are NOT statistically identical
  - Combining does NOT remove dataset bias
  - AD-Auditory is task-based (40Hz auditory), ADFTD/BrainLat are resting-state
  - BrainLat 128-channel data not yet included (needs 128->19 mapping)
  - MCI kept as separate cl

## 7. Save Master Metadata

In [12]:
master_meta.to_csv(COMBINED_META_PATH, index=False)
print(f'Saved master metadata: {COMBINED_META_PATH}')
print(f'Total records: {len(master_meta)}')

print('Combined dataset ready for subject-wise splitting.')

Saved master metadata: /home/admin1/Desktop/SIH_2026_ALL/processed_data/combined/combined_metadata.csv
Total records: 6381
Combined dataset ready for subject-wise splitting.
